# Foundation Models Quickstart: CausalPFN

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/layer6ai-labs/causalfm-survey/blob/main/notebooks/Foundation_models_quickstart.ipynb)

The minimal code to a working causal foundation model: install, fit,
predict; that's it. This runs **CausalPFN** end to end on one simulated,
confounded dataset, calling its native API directly (no wrappers from this
repo), so any cell below can be copied straight into your own project.

Want all three foundation models (CausalPFN, Do-PFN, CausalFM) compared
side by side, with their setup? See
[`Foundation_models_sandbox.ipynb`](Foundation_models_sandbox.ipynb).

In [ ]:
import numpy as np
import torch
from sklearn.model_selection import train_test_split

# Simulated, confounded dataset: a discount email's effect on next-month
# spend. Treatment is more likely for loyal, high-spend customers, so a
# naive treated-vs-control comparison would be biased.
SEED = 42
rng = np.random.default_rng(SEED)
n = 1500

recency, monetary, age = rng.normal(0, 1, (3, n)).astype(np.float32)
X = np.column_stack([recency, monetary, age])

propensity = 1 / (1 + np.exp(-(0.8 * monetary - 0.6 * recency)))
T = rng.binomial(1, propensity).astype(np.float32)

tau_true = (2.0 + 1.5 * recency - 0.75 * age).astype(np.float32)  # ground truth, unobservable in real data
Y0 = (5.0 + 2.0 * monetary - 0.5 * age + rng.normal(0, 1, n)).astype(np.float32)
Y = np.where(T == 1, Y0 + tau_true, Y0).astype(np.float32)

X_train, X_test, T_train, T_test, Y_train, Y_test, tau_train, tau_test = train_test_split(
    X, T, Y, tau_true, test_size=0.3, random_state=SEED
)
true_ate = float(tau_true.mean())
print(f"True ATE (known only in this simulation): {true_ate:.3f}")

In [ ]:
import importlib.util, platform

if importlib.util.find_spec("causalpfn") is None:
    %pip install -q causalpfn  # first run also downloads pretrained weights from HF Hub

device = "cuda" if torch.cuda.is_available() else "cpu"
# Segfaults on Apple Silicon macOS (CPU and MPS) -- a hard process crash,
# not a catchable exception -- so skip there rather than try/except.
skip_apple_silicon = device != "cuda" and platform.system() == "Darwin" and platform.machine() == "arm64"

In [ ]:
if skip_apple_silicon:
    print("Skipping: CausalPFN segfaults on Apple Silicon macOS -- run this on Colab instead.")
else:
    from causalpfn import CATEEstimator, ATEEstimator

    # fit() doesn't train new weights -- it packages (X, T, Y) as "context"
    # for the frozen, pretrained transformer. estimate_* is one forward pass.
    tau_hat = np.asarray(
        CATEEstimator(device=device, verbose=False)
        .fit(X_train, T_train, Y_train)
        .estimate_cate(X_test)
    ).reshape(-1)
    ate_hat = float(np.asarray(
        ATEEstimator(device=device, verbose=False)
        .fit(X_train, T_train, Y_train)
        .estimate_ate()
    ).reshape(-1)[0])

    pehe = float(np.sqrt(np.mean((tau_hat - tau_test) ** 2)))
    print(f"ATE_hat={ate_hat:.3f}  True_ATE={true_ate:.3f}  PEHE={pehe:.3f}")

## Reference output

Since `SEED = 42` fixes the data, and CausalPFN's weights are frozen (no
training), a correct run should reproduce these numbers (from a verified
Colab GPU run) up to minor floating-point differences:

```
True ATE (known only in this simulation): 1.967
ATE_hat=1.911  True_ATE=1.967  PEHE=0.237
```

If your numbers are close to these, the model loaded and ran correctly.